# Qualidade dos Dados
## Objetivo

Neste notebook serão realizados os processos de limpeza, padronização e validação individual dos datasets. Os relacionamentos entre as entidades e os merges serão realizados posteriormente no Notebook 03.

Não será feito neste notebook
- análise de correlação;
- análise de satisfação;
- análise de custos;
- análise de produtividade;
- respostas às perguntas de negócio;
- merges entre datasets;
- criação dos KPIs finais.


In [ ]:
# @title ## 0. Set Up
!pip install dash plotly pandas sqlite-utils

In [ ]:
# @title ## 1. Importação das Bibliotecas
import pandas as pd
import sqlite3
from dash import Dash, dcc, html, Input, Output
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats as st
from IPython.display import Markdown
from IPython.core.display import HTML

import os
import math
import glob
import itertools
import re
import unicodedata
from typing import Optional, Iterable

from google.colab import drive
drive.mount('/content/drive')

# Cor principal do projeto
PRIMARY_COLOR = "#1A306C" # "deep Core blue" ##1A306C rgb:(26, 48, 106)
SECONDARY_COLORS = sns.light_palette(PRIMARY_COLOR, n_colors=5)

# Estilo geral
sns.set_theme(style="whitegrid")

# Tamanho padrão
plt.rcParams['figure.figsize'] = (10, 6)

# Fonte
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# @title ## 2. Carregamentos dos Dados
df_clientes = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/mini_projeto/data/raw/dc_clientes.csv')
df_apontamentos = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/mini_projeto/data/raw/dc_apontamentos.csv')
df_projetos = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/mini_projeto/data/raw/dc_projetos.csv')
df_analistas = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/mini_projeto/data/raw/dc_analistas.csv')
df_satisfacao = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/mini_projeto/data/raw/dc_satisfacao.csv')

In [ ]:
# @title ## 3. Validação dos Dados
print("=" * 60)
print("VALIDAÇÃO DOS DADOS")
print("=" * 60)
print(f"df_clientes: {df_clientes.shape}")
print(f"df_apontamentos: {df_apontamentos.shape}")
print(f"df_projetos: {df_projetos.shape}")
print(f"df_satisfacao: {df_satisfacao.shape}")
print(f"df_analistas: {df_analistas.shape}")

VALIDAÇÃO DOS DADOS
df_clientes: (62, 7)
df_apontamentos: (9879, 6)
df_projetos: (141, 11)
df_satisfacao: (103, 5)
df_analistas: (24, 6)


In [ ]:
# @title ## 4. Funções auxiliares



In [ ]:
# @title ## Padronizar o nome das das colunas
def padronizar_nomes_colunas(df: pd.DataFrame) -> pd.DataFrame:
    """
    Padroniza os nomes das colunas de um DataFrame.

    Regras:
    - remove espaços nas extremidades;
    - converte para minúsculas;
    - remove acentos;
    - substitui espaços por "_";
    - substitui caracteres especiais por "_";
    - remove "_" duplicados.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame de entrada.

    Returns
    -------
    pd.DataFrame
        DataFrame com nomes de colunas padronizados.
    """
    df = df.copy()

    def normalizar_coluna(coluna):
        coluna = str(coluna).strip().lower()

        # Remove acentos
        coluna = unicodedata.normalize("NFKD", coluna)
        coluna = "".join(
            caractere
            for caractere in coluna
            if not unicodedata.combining(caractere)
        )

        # Substitui qualquer grupo de caracteres não alfanuméricos
        # por "_"
        coluna = re.sub(r"[^a-z0-9]+", "_", coluna)

        # Remove "_" no início/fim
        coluna = coluna.strip("_")

        return coluna

    df.columns = [normalizar_coluna(coluna) for coluna in df.columns]

    return df




In [ ]:
# @title ## Remover os espaços extra
def remover_espacos(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove espaços desnecessários dos valores textuais.

    Regras:
    - remove espaços no início e no final;
    - substitui múltiplos espaços internos por um único espaço;
    - preserva valores nulos.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame de entrada.

    Returns
    -------

    pd.DataFrame
        DataFrame com espaços padronizados.
    """
    df = df.copy()

    colunas_texto = df.select_dtypes(include=["object", "string"]).columns

    for coluna in colunas_texto:
        df[coluna] = (
            df[coluna]
            .astype("string")
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )

    return df




In [ ]:
# @title Padronizar as Strings
def padronizar_strings(
    df: pd.DataFrame,
    colunas: Optional[Iterable[str]] = None,
    remover_acentos: bool = False,
    caixa: str = "title"
) -> pd.DataFrame:
    """
    Padroniza valores textuais selecionados.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame de entrada.

    colunas : iterable of str, optional
        Colunas que serão tratadas.
        Se None, todas as colunas textuais serão utilizadas.

    remover_acentos : bool, default=False
        Se True, remove acentos.

    caixa : {"lower", "upper", "title", "preserve"}
        Formato de capitalização.

    Returns
    -------
    pd.DataFrame
        DataFrame com strings padronizadas.
    """
    df = df.copy()

    if colunas is None:
        colunas = df.select_dtypes(
            include=["object", "string"]
        ).columns

    for coluna in colunas:

        serie = (
            df[coluna]
            .astype("string")
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )

        if caixa == "lower":
            serie = serie.str.lower()

        elif caixa == "upper":
            serie = serie.str.upper()

        elif caixa == "title":
            serie = serie.str.title()

        elif caixa == "preserve":
            pass

        else:
            raise ValueError(
                "caixa deve ser: 'lower', 'upper', 'title' ou 'preserve'."
            )

        if remover_acentos:
            serie = serie.map(
                lambda x: (
                    unicodedata.normalize("NFKD", x)
                    .encode("ascii", "ignore")
                    .decode("utf-8")
                    if pd.notna(x)
                    else x
                )
            )

        df[coluna] = serie

    return df



In [ ]:
# @title ## Converter datas com múltiplos formatos


def converter_datas_mistas(serie):
    """
    Converte datas que apresentam múltiplos formatos.

    Formatos suportados:
    - YYYY-MM-DD
    - DD/MM/YYYY
    - MM/DD/YYYY
    - DD-Mon-YYYY
    - DD-Mês-YYYY

    Datas inválidas são convertidas para NaT.
    """

    serie = serie.astype("string").str.strip()

    # ---------------------------------------------------------
    # 1. Padronização dos meses em português
    # ---------------------------------------------------------

    meses_pt = {
        "Jan": "Jan",
        "Fev": "Feb",
        "Mar": "Mar",
        "Abr": "Apr",
        "Mai": "May",
        "Jun": "Jun",
        "Jul": "Jul",
        "Ago": "Aug",
        "Set": "Sep",
        "Out": "Oct",
        "Nov": "Nov",
        "Dez": "Dec"
    }

    for pt, en in meses_pt.items():
        serie = serie.str.replace(
            f"-{pt}-",
            f"-{en}-",
            regex=False
        )

    resultado = pd.Series(
        pd.NaT,
        index=serie.index,
        dtype="datetime64[ns]"
    )

    # ---------------------------------------------------------
    # 2. ISO: YYYY-MM-DD
    # ---------------------------------------------------------

    mascara_iso = serie.str.fullmatch(
        r"\d{4}-\d{2}-\d{2}"
    )

    resultado.loc[mascara_iso] = pd.to_datetime(
        serie.loc[mascara_iso],
        format="%Y-%m-%d",
        errors="coerce"
    )

    # ---------------------------------------------------------
    # 3. Datas com mês textual: DD-Mon-YYYY
    # ---------------------------------------------------------

    mascara_texto = serie.str.fullmatch(
        r"\d{2}-[A-Za-z]{3}-\d{4}"
    )

    resultado.loc[mascara_texto] = pd.to_datetime(
        serie.loc[mascara_texto],
        format="%d-%b-%Y",
        errors="coerce"
    )

    # ---------------------------------------------------------
    # 4. Datas com barra
    # ---------------------------------------------------------

    mascara_barra = serie.str.fullmatch(
        r"\d{2}/\d{2}/\d{4}"
    )

    valores_barra = serie.loc[mascara_barra]

    for indice, valor in valores_barra.items():

        dia, mes, ano = map(int, valor.split("/"))

        # Se o segundo elemento > 12,
        # obrigatoriamente é MM/DD/YYYY.
        if mes > 12:

            resultado.loc[indice] = pd.to_datetime(
                valor,
                format="%m/%d/%Y",
                errors="coerce"
            )

        # Se o primeiro elemento > 12,
        # obrigatoriamente é DD/MM/YYYY.
        elif dia > 12:

            resultado.loc[indice] = pd.to_datetime(
                valor,
                format="%d/%m/%Y",
                errors="coerce"
            )

        else:
            # Quando os dois valores são <= 12,
            # usamos o padrão brasileiro.
            resultado.loc[indice] = pd.to_datetime(
                valor,
                format="%d/%m/%Y",
                errors="coerce"
            )

    return resultado

In [ ]:
# @title ## Converter String para valores monetário

def converter_monetario(
    serie: pd.Series,
    moeda: str = "BRL"
) -> pd.Series:
    """
    Converte valores monetários para formato numérico.

    Aceita formatos como:
    - '150,00'
    - '95.00'
    - 'R$ 1.250,50'
    - '1.250,50'

    Parameters
    ----------
    serie : pd.Series
        Série contendo valores monetários.

    moeda : str, default='BRL'
        Apenas informativo; não altera o resultado numérico.

    Returns
    -------
    pd.Series
        Série numérica do tipo float.
    """

    serie = serie.astype("string").str.strip()

    # Remove símbolos e letras
    serie = serie.str.replace(
        r"[^\d,.\-]",
        "",
        regex=True
    )

    def converter(valor):

        if pd.isna(valor) or valor == "":
            return np.nan

        # Caso tenha ponto e vírgula:
        # assume formato brasileiro: 1.234,56
        if "." in valor and "," in valor:
            valor = valor.replace(".", "").replace(",", ".")

        # Apenas vírgula:
        elif "," in valor:
            valor = valor.replace(",", ".")

        # Apenas ponto:
        # mantém como separador decimal

        try:
            return float(valor)

        except ValueError:
            return np.nan#

    return serie.map(converter).astype("float64")

In [ ]:
# @title Converter números
def converter_numeros(
    serie: pd.Series,
    decimal: str = ","
) -> pd.Series:
    """
    Converte uma série para formato numérico.

    Parameters
    ----------
    serie : pd.Series
        Série de entrada.

    decimal : str, default=
        Separador decimal esperado.

    Returns
    -------
    pd.Series
        Série numérica.
    """

    serie = serie.astype("string").str.strip()

    if decimal == ",":
        serie = serie.str.replace(".", "", regex=False)
        serie = serie.str.replace(",", ".", regex=False)

    elif decimal == ".":
        serie = serie.str.replace(",", "", regex=False)

    else:
        raise ValueError(
            "decimal deve ser ',' ou '.'."
        )

    return pd.to_numeric(
        serie,
        errors="coerce"
    )


In [ ]:
# @title Relatório dos valores nulos
def relatorio_nulos(df: pd.DataFrame) -> pd.DataFrame:
    """
    Gera relatório de valores ausentes.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame analisado.

    Returns
    -------
    pd.DataFrame
        Relatório contendo quantidade e percentual de nulos.
    """

    relatorio = pd.DataFrame({
        "coluna": df.columns,
        "nulos": df.isna().sum().values,
        "percentual_nulos": (
            df.isna().mean().values * 100
        ).round(2)
    })

    return (
        relatorio
        .sort_values(
            "percentual_nulos",
            ascending=False
        )
        .reset_index(drop=True)
    )

In [ ]:
# @title ## Verificação das cardinalidades


def relatorio_cardinalidade(
    df: pd.DataFrame
) -> pd.DataFrame:
    """
    Gera relatório de cardinalidade das colunas.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame analisado.

    Returns
    -------
    pd.DataFrame
        Quantidade e percentual de valores únicos.
    """

    total = len(df)

    relatorio = pd.DataFrame({
        "coluna": df.columns,
        "tipo": df.dtypes.astype(str).values,
        "registros": total,
        "valores_unicos": [
            df[coluna].nunique(dropna=True)
            for coluna in df.columns
        ],
        "percentual_unicos": [
            round(
                df[coluna].nunique(dropna=True) / total * 100,
                2
            )
            if total > 0 else 0
            for coluna in df.columns
        ]
    })

    return relatorio.sort_values(
        "valores_unicos",
        ascending=False
    ).reset_index(drop=True)

In [ ]:
# @title ## 5. Tratamento dos Dados

In [ ]:
# @title ## 5.1 - Dataset dc_clientes

In [ ]:
# @title ## Informações Iniciais do Dataset - dc_clientes
display(Markdown('### **Primeiras Linhas do Dataset**'))
display(df_clientes.head())

display(Markdown('### **Informações do Dataset**'))
display(df_clientes.info())

display(Markdown('### **Resumo Estatístico do Dataset**'))
display(df_clientes.describe().T)

display(Markdown('### **Quantidade de Valores Ausentes**'))
display(df_clientes.isnull().sum())

display(Markdown('### **Quantidade de Valores Duplicados**'))
display(df_clientes.duplicated().sum())

display(Markdown('### **Quantidade de Valores Únicos**'))
display(df_clientes.nunique())

display(Markdown('### **Os tipos de dados**'))
display(df_clientes.dtypes)

### **Primeiras Linhas do Dataset**

,cliente_id,cliente,setor,porte,cidade,uf,data_cadastro
0,CLI001,Umbu S.A.,NaN,Pequeno,Rio de Janeiro,Rio de Janeiro,2021-10-13
1,CLI002,Xingu S.A.,Indústria,Médio,Florianópolis,SC,2021-03-07
2,CLI003,Aurora S.A.,Saúde,Médio,Curitiba,PR,2021-02-24
3,CLI004,Rubi Ltda,Logística,Pequeno,Florianópolis,SC,2024-04-21
4,CLI005,Ipê S.A.,Logística,Pequeno,Belo Horizonte,MG,15/11/2021


### **Informações do Dataset**

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62 entries, 0 to 61
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   cliente_id     62 non-null     object
 1   cliente        62 non-null     object
 2   setor          57 non-null     object
 3   porte          62 non-null     object
 4   cidade         62 non-null     object
 5   uf             62 non-null     object
 6   data_cadastro  62 non-null     object
dtypes: object(7)
memory usage: 3.5+ KB


None

...

np.int64(2)

### **Quantidade de Valores Únicos**

,0
cliente_id,60
cliente,60
setor,17
porte,3
cidade,10
uf,19
data_cadastro,60


### **Os tipos de dados**

,0
cliente_id,object
cliente,object
setor,object
porte,object
cidade,object
uf,object
data_cadastro,object


In [ ]:
# @title ## Padronização das colunas
df_clientes = padronizar_nomes_colunas(df_clientes)
df_clientes = remover_espacos(df_clientes)
df_clientes =padronizar_strings(df_clientes)


In [ ]:
# converter a coluna cliente_id upper
df_clientes['cliente_id'] = df_clientes['cliente_id'].str.upper()
df_clientes.head()

,cliente_id,cliente,setor,porte,cidade,uf,data_cadastro
0,CLI001,Umbu S.A.,<NA>,Pequeno,Rio De Janeiro,Rio De Janeiro,2021-10-13
1,CLI002,Xingu S.A.,Indústria,Médio,Florianópolis,Sc,2021-03-07
2,CLI003,Aurora S.A.,Saúde,Médio,Curitiba,Pr,2021-02-24
3,CLI004,Rubi Ltda,Logística,Pequeno,Florianópolis,Sc,2024-04-21
4,CLI005,Ipê S.A.,Logística,Pequeno,Belo Horizonte,Mg,15/11/2021


In [ ]:
# @title ##  Converter datas
df_clientes['data_cadastro'].unique()


<StringArray>
[ '2021-10-13',  '2021-03-07',  '2021-02-24',  '2024-04-21',  '15/11/2021',
 '06-Jan-2023',  '02/03/2025',  '2022-08-24',  '2024-12-13',  '2025-10-18',
  '2025-09-05',  '01/07/2022', '30-Dez-2023',  '08/03/2024',  '2025-05-09',
  '2021-05-16',  '2023-03-22',  '2022-05-21',  '28/05/2023', '11-Out-2021',
  '11/10/2021',  '2023-02-21',  '2025-10-29',  '2025-04-24',  '2023-07-19',
  '18/11/2025', '27-Nov-2021',  '02/09/2021',  '2025-12-04',  '2025-04-07',
  '2023-12-17',  '2025-03-27',  '07/10/2024', '13-Fev-2021',  '05/26/2021',
  '2022-03-15',  '2022-01-25',  '2023-08-15',  '2025-01-31',  '08/07/2023',
 '26-Nov-2025',  '08/28/2024',  '2023-09-22',  '2021-01-05',  '2025-02-05',
  '2022-03-22',  '28/04/2021', '07-Nov-2023',  '11/01/2025',  '2024-05-02',
  '2022-06-19',  '2021-09-26',  '2025-03-20',  '31/05/2021', '06-Dez-2025',
  '01/18/2024',  '2025-08-01',  '2025-12-28',  '2024-05-23',  '2026-01-29']
Length: 60, dtype: string

In [ ]:
df_clientes["data_cadastro"] = converter_datas_mistas(
    df_clientes["data_cadastro"]
)

In [ ]:
display(Markdown('### **Os tipos de dados**'))
display(df_clientes.dtypes)

### **Os tipos de dados**

,0
cliente_id,string[python]
cliente,string[python]
setor,string[python]
porte,string[python]
cidade,string[python]
uf,string[python]
data_cadastro,datetime64[ns]


**decisão analítica**
> Datas como : 01/07/2022
assumimos que é 01 de julho de 2022 e não 07 de janeiro de 2022
porque existe contextual no dataset em que as datas majoritoariamente em padrão brasileiro tipo 15/11/2021 , 28/05/2023, 31/05/2021

adotamos ocomo regra padrão DD/MM/YYYY

In [ ]:
# @title ## Tratar os dados duplicados
#verificar as linhas duplicadas
df_clientes.duplicated()

,0
0,False
1,False
2,False
3,False
4,False
...,...
57,False
58,False
59,False
60,True


In [ ]:
# Exibir as linhas duplicadas
duplicated_rows = df_clientes[df_clientes.duplicated()]
display(duplicated_rows)

,cliente_id,cliente,setor,porte,cidade,uf,data_cadastro
60,CLI004,Rubi Ltda,Logística,Pequeno,Florianópolis,Sc,2024-04-21
61,CLI028,Rubi Holding,Setor Público,Pequeno,Rio De Janeiro,Rj,2021-09-02


In [ ]:
df_clientes[df_clientes['cliente_id'] =='CLI004']

,cliente_id,cliente,setor,porte,cidade,uf,data_cadastro
3,CLI004,Rubi Ltda,Logística,Pequeno,Florianópolis,Sc,2024-04-21
60,CLI004,Rubi Ltda,Logística,Pequeno,Florianópolis,Sc,2024-04-21


In [ ]:
df_clientes[df_clientes['cliente_id'] =='CLI028']

,cliente_id,cliente,setor,porte,cidade,uf,data_cadastro
27,CLI028,Rubi Holding,Setor Público,Pequeno,Rio De Janeiro,Rj,2021-09-02
61,CLI028,Rubi Holding,Setor Público,Pequeno,Rio De Janeiro,Rj,2021-09-02


In [ ]:
# remover os dados duplicados
df_clientes = df_clientes.drop_duplicates()

In [ ]:
# @title ## Tratar os dados nulls
df_clientes.isna().sum()

,0
cliente_id,0
cliente,0
setor,5
porte,0
cidade,0
uf,0
data_cadastro,0


In [ ]:
dados_nullos  = df_clientes[df_clientes.isnull().any(axis=1)]
display(dados_nullos)


,cliente_id,cliente,setor,porte,cidade,uf,data_cadastro
0,CLI001,Umbu S.A.,<NA>,Pequeno,Rio De Janeiro,Rio De Janeiro,2021-10-13
12,CLI013,Windsor Brasil,<NA>,Médio,São Paulo,Sp,2023-12-30
24,CLI025,Umbu Participações,<NA>,Pequeno,São Paulo,Sp,2023-07-19
36,CLI037,Serena Group,<NA>,Grande,Curitiba,Pr,2022-01-25
48,CLI049,Cristal Ltda,<NA>,Pequeno,São Paulo,Sp,2025-01-11


In [ ]:
df_clientes[df_clientes['cliente'] =='Umbu S.A.']

,cliente_id,cliente,setor,porte,cidade,uf,data_cadastro
0,CLI001,Umbu S.A.,<NA>,Pequeno,Rio De Janeiro,Rio De Janeiro,2021-10-13


In [ ]:
df_clientes[df_clientes['cliente'] =='Umbu Participações']

,cliente_id,cliente,setor,porte,cidade,uf,data_cadastro
24,CLI025,Umbu Participações,<NA>,Pequeno,São Paulo,Sp,2023-07-19


In [ ]:
# Substituir os valores nullos por Não Informado
df_clientes = df_clientes.fillna('Não Informado')

In [ ]:
# @title Verificar os valores da coluna porte
df_clientes['porte'].unique()

<StringArray>
['Pequeno', 'Médio', 'Grande']
Length: 3, dtype: string

In [ ]:
# @title Verificar os valores da coluna cliente
df_clientes['cliente'].unique()

<StringArray>
[              'Umbu S.A.',              'Xingu S.A.',
             'Aurora S.A.',               'Rubi Ltda',
                'Ipê S.A.',   'Granito Participações',
     'Coral Participações',           'Orion Holding',
          'Basalto Brasil',            'Cristal S.A.',
          'Horizonte S.A.',             'Laguna Ltda',
          'Windsor Brasil',              'Xingu Ltda',
         'Windsor Holding',             'Boreal Ltda',
         'Granito Holding',        'Duna Azul Brasil',
           'Xingu Holding',       'Duna Azul Holding',
           'Quartzo Group',               'Umbu Ltda',
            'Tucano Group',           'Aurora Brasil',
      'Umbu Participações',             'Ipê Holding',
         'Quartzo Holding',            'Rubi Holding',
     'Dunas Participações',             'Serena S.A.',
          'Everest Brasil',           'Coral Holding',
            'Xingu Brasil',              'Orion S.A.',
          'Serena Holding',         'Cristal Holdin

In [ ]:
# @title Verificar os valores da coluna cidadde
df_clientes['cidade'].unique()

<StringArray>
['Rio De Janeiro',  'Florianópolis',       'Curitiba', 'Belo Horizonte',
      'São Paulo',      'Fortaleza',       'Salvador',         'Recife',
       'Brasília',   'Porto Alegre']
Length: 10, dtype: string

In [ ]:
# @title Verificar os valores da coluna uf
df_clientes['uf'].unique()

<StringArray>
['Rio De Janeiro',             'Sc',             'Pr',             'Mg',
             'Sp',             'Ce',             'Ba',             'Pe',
             'Rj',       'Brasília',             'Rs',             'Df',
       'Curitiba',         'Recife']
Length: 14, dtype: string

In [ ]:
# @title ## Tratar as colunas uf
df_clientes['uf'] = df_clientes['uf'].str.upper()
df_clientes['uf'].unique()

<StringArray>
['RIO DE JANEIRO',             'SC',             'PR',             'MG',
             'SP',             'CE',             'BA',             'PE',
             'RJ',       'BRASÍLIA',             'RS',             'DF',
       'CURITIBA',         'RECIFE']
Length: 14, dtype: string

In [ ]:
# substituir Rio de janeiro por RJ
df_clientes['uf'] = df_clientes['uf'].replace('RIO DE JANEIRO', 'RJ')
df_clientes['uf'].unique()

<StringArray>
[      'RJ',       'SC',       'PR',       'MG',       'SP',       'CE',
       'BA',       'PE', 'BRASÍLIA',       'RS',       'DF', 'CURITIBA',
   'RECIFE']
Length: 13, dtype: string

In [ ]:
# substituir BRASÍLIA por DF
df_clientes['uf'] = df_clientes['uf'].replace('BRASÍLIA', 'DF')
df_clientes['uf'].unique()

<StringArray>
[      'RJ',       'SC',       'PR',       'MG',       'SP',       'CE',
       'BA',       'PE',       'DF',       'RS', 'CURITIBA',   'RECIFE']
Length: 12, dtype: string

In [ ]:
# substituir CURITIBA por PR
df_clientes['uf'] = df_clientes['uf'].replace('CURITIBA', 'PR')
df_clientes['uf'].unique()

<StringArray>
['RJ', 'SC', 'PR', 'MG', 'SP', 'CE', 'BA', 'PE', 'DF', 'RS', 'RECIFE']
Length: 11, dtype: string

In [ ]:
# substituir RECIFE POR PE
df_clientes['uf'] = df_clientes['uf'].replace('RECIFE', 'PE')
df_clientes['uf'].unique()

<StringArray>
['RJ', 'SC', 'PR', 'MG', 'SP', 'CE', 'BA', 'PE', 'DF', 'RS']
Length: 10, dtype: string

In [ ]:
# VERIFICAR OS CLIENTES DE CADA ESTADO
df_clientes.groupby('uf')['cidade'].value_counts()

,,count
uf,cidade,
BA,Salvador,6
CE,Fortaleza,5
DF,Brasília,5
MG,Belo Horizonte,5
PE,Recife,3
PR,Curitiba,10
RJ,Rio De Janeiro,4
RS,Porto Alegre,4
SC,Florianópolis,7


In [ ]:
# @title ##  5.2 - Dataset df_analistas

In [ ]:
# @title ## Informações Iniciais do Dataset - df_analistas
display(Markdown('### **Primeiras Linhas do Dataset**'))
display(df_analistas.head())

display(Markdown('### **Informações do Dataset**'))
display(df_analistas.info())

display(Markdown('### **Resumo Estatístico do Dataset**'))
display(df_analistas.describe().T)

display(Markdown('### **Quantidade de Valores Ausentes**'))
display(df_analistas.isnull().sum())

display(Markdown('### **Quantidade de Valores Duplicados**'))
display(df_analistas.duplicated().sum())

display(Markdown('### **Quantidade de Valores Únicos**'))
display(df_analistas.nunique())

display(Markdown('### **Os tipos de dados**'))
display(df_analistas.dtypes)

### **Primeiras Linhas do Dataset**

,analista_id,analista,squad,senioridade,custo_hora,data_admissao
0,ANL001,Ana Barbosa,ALPHA,Senior,"150,00",2021-12-07
1,ANL002,Bruno Ipiranga,Bravo,Pleno,95.00,2021-08-30
2,ANL003,Carla Esteves,Charlie,Junior,55.00,2024-12-26
3,ANL004,Diego Freitas,Delta,Especialista,210.00,2025-07-28
4,ANL005,Elisa Werneck,Echo,Pleno,95.00,24/10/2023


### **Informações do Dataset**

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   analista_id    24 non-null     object
 1   analista       24 non-null     object
 2   squad          24 non-null     object
 3   senioridade    24 non-null     object
 4   custo_hora     24 non-null     object
 5   data_admissao  24 non-null     object
dtypes: object(6)
memory usage: 1.3+ KB


None

...

np.int64(0)

### **Quantidade de Valores Únicos**

,0
analista_id,24
analista,24
squad,9
senioridade,4
custo_hora,7
data_admissao,24


### **Os tipos de dados**

,0
analista_id,object
analista,object
squad,object
senioridade,object
custo_hora,object
data_admissao,object


In [ ]:
# @title ## Padronizar o nome das colunas
df_analistas = padronizar_nomes_colunas(df_analistas)
df_analistas = remover_espacos(df_analistas)
df_analistas = padronizar_strings(df_analistas)

In [ ]:
# converter a coluna analista_id upper
df_analistas['analista_id'] = df_analistas['analista_id'].str.upper()
df_analistas.head()

,analista_id,analista,squad,senioridade,custo_hora,data_admissao
0,ANL001,Ana Barbosa,Alpha,Senior,"150,00",2021-12-07
1,ANL002,Bruno Ipiranga,Bravo,Pleno,95.00,2021-08-30
2,ANL003,Carla Esteves,Charlie,Junior,55.00,2024-12-26
3,ANL004,Diego Freitas,Delta,Especialista,210.00,2025-07-28
4,ANL005,Elisa Werneck,Echo,Pleno,95.00,24/10/2023


In [ ]:
# verificar os valores unicos da coluna analista
df_analistas['analista'].unique()

<StringArray>
[       'Ana Barbosa',     'Bruno Ipiranga',      'Carla Esteves',
      'Diego Freitas',      'Elisa Werneck',      'Felipe Duarte',
   'Gabriela Werneck',     'Henrique Lopes',      'Isabela Nunes',
         'João Lopes',     'Karina Barbosa',    'Lucas Henriques',
    'Mariana Ribeiro',      'Nelson Xavier',   'Olívia Henriques',
      'Paulo Freitas',      'Queila Xavier',       'Rafael Nunes',
       'Sofia Xavier',     'Thiago Freitas',     'Úrsula Marques',
 'Vinícius Henriques',     'Wanda Oliveira',     'Yuri Henriques']
Length: 24, dtype: string

In [ ]:
# verificar os valores unicos da coluna squad
df_analistas['squad'].unique()

<StringArray>
['Alpha', 'Bravo', 'Charlie', 'Delta', 'Echo', 'Foxtrot']
Length: 6, dtype: string

In [ ]:
# verificar os valores unicos da coluna senioridade
df_analistas['senioridade'].unique()


<StringArray>
['Senior', 'Pleno', 'Junior', 'Especialista']
Length: 4, dtype: string

In [ ]:
# tratar a coluna custo_hora para float
df_analistas['custo_hora'] = converter_monetario(
    df_analistas['custo_hora']
)
#

In [ ]:
df_analistas.dtypes

,0
analista_id,string[python]
analista,string[python]
squad,string[python]
senioridade,string[python]
custo_hora,float64
data_admissao,string[python]


In [ ]:
# tratar data_admissao
df_analistas['data_admissao'] = converter_datas_mistas(
    df_analistas['data_admissao']
)

In [ ]:
df_analistas.dtypes

,0
analista_id,string[python]
analista,string[python]
squad,string[python]
senioridade,string[python]
custo_hora,float64
data_admissao,datetime64[ns]


In [ ]:
df_analistas['data_admissao'].unique()

<DatetimeArray>
['2021-12-07 00:00:00', '2021-08-30 00:00:00', '2024-12-26 00:00:00',
 '2025-07-28 00:00:00', '2023-10-24 00:00:00', '2021-11-02 00:00:00',
 '2022-02-04 00:00:00', '2024-09-05 00:00:00', '2022-02-17 00:00:00',
 '2025-11-17 00:00:00', '2023-06-03 00:00:00', '2025-02-25 00:00:00',
 '2023-09-11 00:00:00', '2022-04-13 00:00:00', '2022-04-29 00:00:00',
 '2023-09-23 00:00:00', '2023-04-12 00:00:00', '2025-11-29 00:00:00',
 '2025-12-17 00:00:00', '2025-10-30 00:00:00', '2021-08-19 00:00:00',
 '2022-07-14 00:00:00', '2023-05-18 00:00:00', '2022-08-31 00:00:00']
Length: 24, dtype: datetime64[ns]

In [ ]:
df_analistas.head()

,analista_id,analista,squad,senioridade,custo_hora,data_admissao
0,ANL001,Ana Barbosa,Alpha,Senior,150.0,2021-12-07
1,ANL002,Bruno Ipiranga,Bravo,Pleno,95.0,2021-08-30
2,ANL003,Carla Esteves,Charlie,Junior,55.0,2024-12-26
3,ANL004,Diego Freitas,Delta,Especialista,210.0,2025-07-28
4,ANL005,Elisa Werneck,Echo,Pleno,95.0,2023-10-24


In [ ]:
# @title ## 5.3 - Dataset df_projetos

In [ ]:
# @title ## Informações Iniciais do Dataset - df_projetos
display(Markdown('### **Primeiras Linhas do Dataset**'))
display(df_projetos.head())

display(Markdown('### **Informações do Dataset**'))
display(df_projetos.info())

display(Markdown('### **Resumo Estatístico do Dataset**'))
display(df_projetos.describe().T)

display(Markdown('### **Quantidade de Valores Ausentes**'))
display(df_projetos.isnull().sum())

display(Markdown('### **Quantidade de Valores Duplicados**'))
display(df_projetos.duplicated().sum())

display(Markdown('### **Quantidade de Valores Únicos**'))
display(df_projetos.nunique())

display(Markdown('### **Os tipos de dados**'))
display(df_projetos.dtypes)

### **Primeiras Linhas do Dataset**

,projeto_id,cliente_id,nome_projeto,tipo_servico,squad,status,data_inicio,data_fim_prevista,data_fim_real,horas_vendidas,valor_contrato
0,PRJ0001,CLI013,Dashboard BI - Windsor,Dashboard B.I.,Delta,CONCLUÍDO,2024-03-19,2024-05-06,08/05/2024,288,NaN
1,PRJ0002,CLI022,Diagnóstico de Dados - Umbu,Diagnóstico de Dados,Alpha,Cancelado,2024-10-01,2024-10-28,NaN,89,24044.51
2,PRJ0003,CLI051,Pipeline de Dados - Cristal,Pipeline de Dados,Charlie,Concluído,2025-06-16,20/10/2025,10/15/2025,423,91525.58
3,PRJ0004,CLI060,Dashboard BI - Kairós,Dashboard BI,Echo,Concluído,2024-03-19,25-abr-2024,2024-04-26,220,47409.01
4,PRJ0005,CLI020,Modelo Preditivo - Duna,Modelo Preditivo,Echo,Concluído,09/12/2024,02/27/2025,2025-03-12,338,"61.970,49"


### **Informações do Dataset**

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 141 entries, 0 to 140
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   projeto_id         141 non-null    object
 1   cliente_id         141 non-null    object
 2   nome_projeto       141 non-null    object
 3   tipo_servico       141 non-null    object
 4   squad              141 non-null    object
 5   status             141 non-null    object
 6   data_inicio        141 non-null    object
 7   data_fim_prevista  141 non-null    object
 8   data_fim_real      128 non-null    object
 9   horas_vendidas     141 non-null    int64 
 10  valor_contrato     134 non-null    object
dtypes: int64(1), object(10)
memory usage: 12.2+ KB


None

...

np.int64(1)

### **Quantidade de Valores Únicos**

,0
projeto_id,140
cliente_id,54
nome_projeto,87
tipo_servico,20
squad,6
status,5
data_inicio,132
data_fim_prevista,131
data_fim_real,117
horas_vendidas,124


### **Os tipos de dados**

,0
projeto_id,object
cliente_id,object
nome_projeto,object
tipo_servico,object
squad,object
status,object
data_inicio,object
data_fim_prevista,object
data_fim_real,object
horas_vendidas,int64


In [ ]:
# @title ## Padronizar o nome das colunas
df_projetos = padronizar_nomes_colunas(df_projetos)
df_projetos = remover_espacos(df_projetos)
df_projetos = padronizar_strings(df_projetos)

In [ ]:
# converter a coluna analista_id upper
df_projetos['projeto_id'] = df_projetos['projeto_id'].str.upper()
df_projetos['cliente_id'] = df_projetos['cliente_id'].str.upper()
df_projetos.head()

,projeto_id,cliente_id,nome_projeto,tipo_servico,squad,status,data_inicio,data_fim_prevista,data_fim_real,horas_vendidas,valor_contrato
0,PRJ0001,CLI013,Dashboard Bi - Windsor,Dashboard B.I.,Delta,Concluído,2024-03-19,2024-05-06,08/05/2024,288,<NA>
1,PRJ0002,CLI022,Diagnóstico De Dados - Umbu,Diagnóstico De Dados,Alpha,Cancelado,2024-10-01,2024-10-28,<NA>,89,24044.51
2,PRJ0003,CLI051,Pipeline De Dados - Cristal,Pipeline De Dados,Charlie,Concluído,2025-06-16,20/10/2025,10/15/2025,423,91525.58
3,PRJ0004,CLI060,Dashboard Bi - Kairós,Dashboard Bi,Echo,Concluído,2024-03-19,25-Abr-2024,2024-04-26,220,47409.01
4,PRJ0005,CLI020,Modelo Preditivo - Duna,Modelo Preditivo,Echo,Concluído,09/12/2024,02/27/2025,2025-03-12,338,"61.970,49"


In [ ]:
# @title ## Converter string em datetime
# converter string data em datetime
df_projetos['data_inicio'] = converter_datas_mistas(
    df_projetos['data_inicio']
)
df_projetos['data_fim_real'] = converter_datas_mistas(
    df_projetos['data_fim_real']
)
df_projetos['data_fim_prevista'] = converter_datas_mistas(
    df_projetos['data_fim_prevista']
)

In [ ]:
# @title ## Tratamento dos valores monetário
# tratar a coluna valor_contrato
df_projetos['valor_contrato'] = converter_monetario(
    df_projetos['valor_contrato']
)

In [ ]:
df_projetos.dtypes

,0
projeto_id,string[python]
cliente_id,string[python]
nome_projeto,string[python]
tipo_servico,string[python]
squad,string[python]
status,string[python]
data_inicio,datetime64[ns]
data_fim_prevista,datetime64[ns]
data_fim_real,datetime64[ns]
horas_vendidas,int64


In [ ]:
# @title ## Tratar os dados duplicados
# verificar os dados duplicados
df_projetos.duplicated().sum()

np.int64(1)

In [ ]:
# Exibir as linhas duplicadas
duplicated_rows = df_projetos[df_projetos.duplicated()]
display(duplicated_rows)

,projeto_id,cliente_id,nome_projeto,tipo_servico,squad,status,data_inicio,data_fim_prevista,data_fim_real,horas_vendidas,valor_contrato
140,PRJ0010,CLI001,Modelo Preditivo - Umbu,Modelo Preditivo,Alpha,Concluído,2024-07-08,2024-10-22,2024-10-21,578,98986.73


In [ ]:
# remover os dados duplicados
df_projetos = df_projetos.drop_duplicates()

In [ ]:
# verificar os dados nullos
df_projetos.isna().sum()


,0
projeto_id,0
cliente_id,0
nome_projeto,0
tipo_servico,0
squad,0
status,0
data_inicio,0
data_fim_prevista,0
data_fim_real,13
horas_vendidas,0


In [ ]:
# @title ## Tratamentos dos dados nulos
# verificar os valores nulls do valor de contrato
nulls_valor_contrato = df_projetos[df_projetos['valor_contrato'].isnull()]
display(nulls_valor_contrato)

,projeto_id,cliente_id,nome_projeto,tipo_servico,squad,status,data_inicio,data_fim_prevista,data_fim_real,horas_vendidas,valor_contrato
0,PRJ0001,CLI013,Dashboard Bi - Windsor,Dashboard B.I.,Delta,Concluído,2024-03-19,2024-05-06,2024-05-08,288,NaN
23,PRJ0024,CLI044,Pipeline De Dados - Pantanal,Pipeline De Dados,Echo,Concluído,2025-08-11,2025-12-03,2025-01-12,612,NaN
46,PRJ0047,CLI032,Modelo Preditivo - Coral,Modelo Preditivo,Foxtrot,Concluído,2024-08-05,2024-11-19,2024-12-12,406,NaN
69,PRJ0070,CLI041,Modelo Preditivo - Everest,Modelo Preditivo,Delta,Concluído,2025-03-03,2025-08-11,2025-08-18,538,NaN
92,PRJ0093,CLI012,Modelo Preditivo - Laguna,Modelo Preditivo,Bravo,Concluído,2024-08-30,2024-11-21,2024-11-18,374,NaN
115,PRJ0116,CLI009,Diagnóstico De Dados - Basalto,Diagnóstico De Dados,Alpha,Concluído,2025-05-05,2025-06-02,2025-06-05,133,NaN
138,PRJ0139,CLI023,Modelo Preditivo - Tucano,Modelo Preditivo,Echo,Concluído,2024-09-20,2024-12-19,2024-12-16,325,NaN


In [ ]:
# Atribuir o valor com o mesmo tipo de servico
df_projetos[(df_projetos['tipo_servico'] =='Modelo Preditivo') & (df_projetos['squad'] =='Foxtrot')]

,projeto_id,cliente_id,nome_projeto,tipo_servico,squad,status,data_inicio,data_fim_prevista,data_fim_real,horas_vendidas,valor_contrato
33,PRJ0034,CLI009,Modelo Preditivo - Basalto,Modelo Preditivo,Foxtrot,Concluído,2024-01-22,2024-05-20,2024-06-24,565,98617.32
46,PRJ0047,CLI032,Modelo Preditivo - Coral,Modelo Preditivo,Foxtrot,Concluído,2024-08-05,2024-11-19,2024-12-12,406,NaN


In [ ]:
 # calcular o valor da hora do tipo de projeto para a mesma squad
valor_hora = df_projetos[(df_projetos['tipo_servico'] =='Modelo Preditivo') &
    (df_projetos['squad'] =='Foxtrot')]['valor_contrato'] / df_projetos[(df_projetos['tipo_servico'] =='Modelo Preditivo') &
    (df_projetos['squad'] =='Foxtrot')]['horas_vendidas']

valor_hora

,0
33,174.543929
46,NaN


In [ ]:
# atribuir o valor_hora * horas_vendidas para a linha 46, usando o valor de hora do projeto similar (índice 33)
df_projetos.loc[46, 'valor_contrato'] = (valor_hora.loc[33] * df_projetos.loc[46, 'horas_vendidas']).round(2)

In [ ]:
df_projetos[(df_projetos['tipo_servico'] =='Modelo Preditivo') & (df_projetos['squad'] =='Foxtrot')]

,projeto_id,cliente_id,nome_projeto,tipo_servico,squad,status,data_inicio,data_fim_prevista,data_fim_real,horas_vendidas,valor_contrato
33,PRJ0034,CLI009,Modelo Preditivo - Basalto,Modelo Preditivo,Foxtrot,Concluído,2024-01-22,2024-05-20,2024-06-24,565,98617.32
46,PRJ0047,CLI032,Modelo Preditivo - Coral,Modelo Preditivo,Foxtrot,Concluído,2024-08-05,2024-11-19,2024-12-12,406,70864.84


In [ ]:
# Atribuir o valor com o mesmo tipo de servico
df_projetos[(df_projetos['tipo_servico'] =='Modelo Preditivo') & (df_projetos['squad'] =='Delta')]

,projeto_id,cliente_id,nome_projeto,tipo_servico,squad,status,data_inicio,data_fim_prevista,data_fim_real,horas_vendidas,valor_contrato
24,PRJ0025,CLI001,Modelo Preditivo - Umbu,Modelo Preditivo,Delta,Concluído,2025-01-13,2025-04-10,2025-04-14,430,66769.09
64,PRJ0065,CLI020,Modelo Preditivo - Duna,Modelo Preditivo,Delta,Concluído,2026-01-26,2026-03-25,2026-03-23,197,35258.80
69,PRJ0070,CLI041,Modelo Preditivo - Everest,Modelo Preditivo,Delta,Concluído,2025-03-03,2025-08-11,2025-08-18,538,NaN
81,PRJ0082,CLI014,Modelo Preditivo - Xingu,Modelo Preditivo,Delta,Concluído,2025-03-06,2025-06-16,2025-07-17,411,79900.27
88,PRJ0089,CLI014,Modelo Preditivo - Xingu,Modelo Preditivo,Delta,Concluído,2024-06-17,2024-09-20,2024-09-23,394,68036.82


In [ ]:
df_projetos[(df_projetos['tipo_servico'] =='Modelo Preditivo') & (df_projetos['squad'] =='Delta')].describe()

,data_inicio,data_fim_prevista,data_fim_real,horas_vendidas,valor_contrato
count,5,5,5,5.000000,4.0000
mean,2025-03-07 19:12:00,2025-06-16 09:36:00,2025-06-25 00:00:00,394.000000,62491.2450
min,2024-06-17 00:00:00,2024-09-20 00:00:00,2024-09-23 00:00:00,197.000000,35258.8000
25%,2025-01-13 00:00:00,2025-04-10 00:00:00,2025-04-14 00:00:00,394.000000,58891.5175
50%,2025-03-03 00:00:00,2025-06-16 00:00:00,2025-07-17 00:00:00,411.000000,67402.9550
75%,2025-03-06 00:00:00,2025-08-11 00:00:00,2025-08-18 00:00:00,430.000000,71002.6825
max,2026-01-26 00:00:00,2026-03-25 00:00:00,2026-03-23 00:00:00,538.000000,79900.2700
std,NaN,NaN,NaN,123.622409,19093.9220


In [ ]:
valor_hora = df_projetos[(df_projetos['tipo_servico'] =='Modelo Preditivo') &
    (df_projetos['squad'] =='Delta')]['valor_contrato'] / df_projetos[(df_projetos['tipo_servico'] =='Modelo Preditivo') &
    (df_projetos['squad'] =='Delta')]['horas_vendidas']

valor_hora

,0
24,155.276953
64,178.978680
69,NaN
81,194.404550
88,172.682284


In [ ]:
df_projetos.loc[69, 'valor_contrato'] = (valor_hora.loc[64] * df_projetos.loc[69, 'horas_vendidas']).round(2)

In [ ]:
nulls_valor_contrato = df_projetos[df_projetos['valor_contrato'].isnull()]
display(nulls_valor_contrato)

,projeto_id,cliente_id,nome_projeto,tipo_servico,squad,status,data_inicio,data_fim_prevista,data_fim_real,horas_vendidas,valor_contrato
0,PRJ0001,CLI013,Dashboard Bi - Windsor,Dashboard B.I.,Delta,Concluído,2024-03-19,2024-05-06,2024-05-08,288,NaN
23,PRJ0024,CLI044,Pipeline De Dados - Pantanal,Pipeline De Dados,Echo,Concluído,2025-08-11,2025-12-03,2025-01-12,612,NaN
92,PRJ0093,CLI012,Modelo Preditivo - Laguna,Modelo Preditivo,Bravo,Concluído,2024-08-30,2024-11-21,2024-11-18,374,NaN
115,PRJ0116,CLI009,Diagnóstico De Dados - Basalto,Diagnóstico De Dados,Alpha,Concluído,2025-05-05,2025-06-02,2025-06-05,133,NaN
138,PRJ0139,CLI023,Modelo Preditivo - Tucano,Modelo Preditivo,Echo,Concluído,2024-09-20,2024-12-19,2024-12-16,325,NaN


In [ ]:
#  Atribuir o valor com o mesmo tipo de servico e squad
df_projetos[(df_projetos['tipo_servico'] =='Modelo Preditivo') & (df_projetos['squad'] =='Bravo')]

,projeto_id,cliente_id,nome_projeto,tipo_servico,squad,status,data_inicio,data_fim_prevista,data_fim_real,horas_vendidas,valor_contrato
58,PRJ0059,CLI012,Modelo Preditivo - Laguna,Modelo Preditivo,Bravo,Concluído,2025-02-03,2025-03-31,2025-12-05,269,41412.12
92,PRJ0093,CLI012,Modelo Preditivo - Laguna,Modelo Preditivo,Bravo,Concluído,2024-08-30,2024-11-21,2024-11-18,374,NaN
122,PRJ0123,CLI052,Modelo Preditivo - Granito,Modelo Preditivo,Bravo,Concluído,2025-04-21,2025-06-30,2025-08-25,212,39390.08


In [ ]:
valor_hora = df_projetos[(df_projetos['tipo_servico'] =='Modelo Preditivo') &
    (df_projetos['squad'] =='Bravo')]['valor_contrato'] / df_projetos[(df_projetos['tipo_servico'] =='Modelo Preditivo') &
    (df_projetos['squad'] =='Bravo')]['horas_vendidas']

valor_hora

,0
58,153.948401
92,NaN
122,185.802264


In [ ]:
df_projetos.loc[92, 'valor_contrato'] = (valor_hora.loc[122] * df_projetos.loc[92, 'horas_vendidas']).round(2)

In [ ]:
df_projetos[(df_projetos['tipo_servico'] =='Modelo Preditivo') & (df_projetos['squad'] =='Echo')]

,projeto_id,cliente_id,nome_projeto,tipo_servico,squad,status,data_inicio,data_fim_prevista,data_fim_real,horas_vendidas,valor_contrato
4,PRJ0005,CLI020,Modelo Preditivo - Duna,Modelo Preditivo,Echo,Concluído,2024-12-09,2025-02-27,2025-03-12,338,61970.49
125,PRJ0126,CLI024,Modelo Preditivo - Aurora,Modelo Preditivo,Echo,Concluído,2026-01-21,2026-04-06,2026-06-02,316,60058.14
138,PRJ0139,CLI023,Modelo Preditivo - Tucano,Modelo Preditivo,Echo,Concluído,2024-09-20,2024-12-19,2024-12-16,325,NaN


In [ ]:
valor_hora = df_projetos[(df_projetos['tipo_servico'] =='Modelo Preditivo') &
    (df_projetos['squad'] =='Echo')]['valor_contrato'] / df_projetos[(df_projetos['tipo_servico'] =='Modelo Preditivo') &
    (df_projetos['squad'] =='Echo')]['horas_vendidas']

valor_hora

,0
4,183.344645
125,190.057405
138,NaN


In [ ]:
df_projetos.loc[138, 'valor_contrato'] = (valor_hora.loc[125] * df_projetos.loc[138, 'horas_vendidas']).round(2)

In [ ]:
# verificar os valores nulls do valor de contrato
nulls_valor_contrato = df_projetos[df_projetos['valor_contrato'].isnull()]
display(nulls_valor_contrato)

,projeto_id,cliente_id,nome_projeto,tipo_servico,squad,status,data_inicio,data_fim_prevista,data_fim_real,horas_vendidas,valor_contrato
0,PRJ0001,CLI013,Dashboard Bi - Windsor,Dashboard B.I.,Delta,Concluído,2024-03-19,2024-05-06,2024-05-08,288,NaN
23,PRJ0024,CLI044,Pipeline De Dados - Pantanal,Pipeline De Dados,Echo,Concluído,2025-08-11,2025-12-03,2025-01-12,612,NaN
115,PRJ0116,CLI009,Diagnóstico De Dados - Basalto,Diagnóstico De Dados,Alpha,Concluído,2025-05-05,2025-06-02,2025-06-05,133,NaN


In [ ]:
df_projetos[(df_projetos['tipo_servico'] =='Dashboard B.I.') & (df_projetos['squad'] =='Delta')]

,projeto_id,cliente_id,nome_projeto,tipo_servico,squad,status,data_inicio,data_fim_prevista,data_fim_real,horas_vendidas,valor_contrato
0,PRJ0001,CLI013,Dashboard Bi - Windsor,Dashboard B.I.,Delta,Concluído,2024-03-19,2024-05-06,2024-05-08,288,NaN
18,PRJ0019,CLI017,Dashboard Bi - Granito,Dashboard B.I.,Delta,Concluído,2024-12-09,2025-02-18,2025-02-14,287,58782.26


In [ ]:
valor_hora = df_projetos[(df_projetos['tipo_servico'] =='Dashboard B.I.') &
    (df_projetos['squad'] =='Delta')]['valor_contrato'] / df_projetos[(df_projetos['tipo_servico'] =='Dashboard B.I.') &
    (df_projetos['squad'] =='Delta')]['horas_vendidas']

valor_hora

,0
0,NaN
18,204.816237


In [ ]:
df_projetos.loc[0, 'valor_contrato'] = (valor_hora.loc[18] * df_projetos.loc[0, 'horas_vendidas']).round(2)

In [ ]:
df_projetos[(df_projetos['tipo_servico'] =='Pipeline De Dados') & (df_projetos['squad'] =='Echo')]

,projeto_id,cliente_id,nome_projeto,tipo_servico,squad,status,data_inicio,data_fim_prevista,data_fim_real,horas_vendidas,valor_contrato
23,PRJ0024,CLI044,Pipeline De Dados - Pantanal,Pipeline De Dados,Echo,Concluído,2025-08-11,2025-12-03,2025-01-12,612,NaN
50,PRJ0051,CLI060,Pipeline De Dados - Kairós,Pipeline De Dados,Echo,Concluído,2024-10-23,2024-12-26,2024-12-27,336,77022.28
61,PRJ0062,CLI023,Pipeline De Dados - Tucano,Pipeline De Dados,Echo,Concluído,2024-11-05,2025-02-27,2025-02-25,351,84086.09
77,PRJ0078,CLI024,Pipeline De Dados - Aurora,Pipeline De Dados,Echo,Concluído,2025-06-06,2025-10-21,2025-10-24,503,93831.25
83,PRJ0084,CLI009,Pipeline De Dados - Basalto,Pipeline De Dados,Echo,Concluído,2025-05-08,2025-09-23,2025-09-22,230,49493.22
102,PRJ0103,CLI029,Pipeline De Dados - Dunas,Pipeline De Dados,Echo,Concluído,2025-01-14,2025-07-05,2025-05-05,467,101146.75
114,PRJ0115,CLI026,Pipeline De Dados - Ipê,Pipeline De Dados,Echo,Concluído,2024-04-01,2024-07-04,2024-07-15,391,75911.35


In [ ]:
valor_hora = df_projetos[(df_projetos['tipo_servico'] =='Pipeline De Dados') &
    (df_projetos['squad'] =='Echo')]['valor_contrato'] / df_projetos[(df_projetos['tipo_servico'] =='Pipeline De Dados') &
    (df_projetos['squad'] =='Echo')]['horas_vendidas']

valor_hora

,0
23,NaN
50,229.232976
61,239.561510
77,186.543241
83,215.187913
102,216.588330
114,194.146675


In [ ]:
df_projetos.loc[23, 'valor_contrato'] = (valor_hora.loc[61] * df_projetos.loc[23, 'horas_vendidas']).round(2)

In [ ]:
df_projetos[(df_projetos['tipo_servico'] =='Diagnóstico De Dados') & (df_projetos['squad'] =='Alpha')]

,projeto_id,cliente_id,nome_projeto,tipo_servico,squad,status,data_inicio,data_fim_prevista,data_fim_real,horas_vendidas,valor_contrato
1,PRJ0002,CLI022,Diagnóstico De Dados - Umbu,Diagnóstico De Dados,Alpha,Cancelado,2024-10-01,2024-10-28,NaT,89,24044.51
66,PRJ0067,CLI052,Diagnóstico De Dados - Granito,Diagnóstico De Dados,Alpha,Concluído,2024-05-31,2024-07-04,2024-07-08,179,49588.63
115,PRJ0116,CLI009,Diagnóstico De Dados - Basalto,Diagnóstico De Dados,Alpha,Concluído,2025-05-05,2025-06-02,2025-06-05,133,NaN


In [ ]:
valor_hora = df_projetos[(df_projetos['tipo_servico'] =='Diagnóstico De Dados') &
    (df_projetos['squad'] =='Alpha')]['valor_contrato'] / df_projetos[(df_projetos['tipo_servico'] =='Diagnóstico De Dados') &
    (df_projetos['squad'] =='Alpha')]['horas_vendidas']

valor_hora

,0
1,270.163034
66,277.031453
115,NaN


In [ ]:
df_projetos.loc[115, 'valor_contrato'] = (valor_hora.loc[66] * df_projetos.loc[115, 'horas_vendidas']).round(2)

In [ ]:
nulls_data_fim_real = df_projetos[df_projetos['data_fim_real'].isnull()]
display(nulls_data_fim_real)


,projeto_id,cliente_id,nome_projeto,tipo_servico,squad,status,data_inicio,data_fim_prevista,data_fim_real,horas_vendidas,valor_contrato
1,PRJ0002,CLI022,Diagnóstico De Dados - Umbu,Diagnóstico De Dados,Alpha,Cancelado,2024-10-01,2024-10-28,NaT,89,24044.51
7,PRJ0008,CLI041,Data Warehouse - Everest,Data Warehouse,Echo,Em Andamento,2026-02-16,2026-09-04,NaT,743,133841.27
11,PRJ0012,CLI054,Data Warehouse - Aurora,Data Warehouse,Echo,Em Andamento,2026-02-09,2026-07-13,NaT,856,192040.56
35,PRJ0036,CLI020,Data Warehouse - Duna,Data Warehouse,Alpha,Em Andamento,2026-03-02,2026-08-03,NaT,654,118710.70
37,PRJ0038,CLI041,Diagnóstico De Dados - Everest,Diagnóstico De Dados,Bravo,Cancelado,2024-06-17,2024-07-22,NaT,127,32109.95
62,PRJ0063,CLI048,Modelo Preditivo - Pantanal,Modelo Preditivo,Alpha,Em Andamento,2026-11-05,2026-08-04,NaT,490,91756.72
80,PRJ0081,CLI030,Data Warehouse - Serena,Data Warehouse,Echo,Em Andamento,2026-03-30,2026-10-15,NaT,1019,202435.52
100,PRJ0101,CLI036,Pipeline De Dados - Cristal,Pipeline De Dados,Foxtrot,Em Andamento,2026-04-16,2026-09-02,NaT,658,120059.58
105,PRJ0106,CLI006,Data Warehouse - Granito,Data Warehouse,Charlie,Em Andamento,2026-03-30,2026-11-30,NaT,972,190237.28
119,PRJ0120,CLI024,Data Warehouse - Aurora,Data Warehouse,Alpha,Em Andamento,2026-03-23,2026-08-27,NaT,938,192219.19


**Tratamento de valores ausentes em data_fim_real**

Foram identificados valores ausentes (NaT) na variável data_fim_real. A análise preliminar dos registros indica que esses valores estão associados a projetos classificados como "Em Andamento" ou "Cancelado", situações nas quais uma data de conclusão real pode não existir.

Dessa forma, os valores ausentes foram preservados como NaT, sem aplicação de imputação, uma vez que substituir esses valores pela data prevista, pela data atual ou por uma medida estatística introduziria uma informação que não está presente na fonte original.

Essa decisão será posteriormente validada durante a etapa de análise de qualidade e modelagem dos relacionamentos entre os datasets.

In [ ]:
inconsistencias = df_projetos[
    df_projetos["data_inicio"]
    > df_projetos["data_fim_prevista"]
]

inconsistencias[
    [
        "projeto_id",
        "status",
        "data_inicio",
        "data_fim_prevista",
        "data_fim_real"
    ]
]

,projeto_id,status,data_inicio,data_fim_prevista,data_fim_real
13,PRJ0014,Concluído,2024-12-06,2024-10-14,2024-11-14
48,PRJ0049,Concluído,2026-12-01,2026-03-04,2026-03-02
62,PRJ0063,Em Andamento,2026-11-05,2026-08-04,NaT
116,PRJ0117,Concluído,2025-07-22,2025-06-08,2025-08-04
118,PRJ0119,Concluído,2024-09-01,2024-04-30,2024-05-31
123,PRJ0124,Concluído,2024-09-23,2024-08-11,2024-11-04


> Inconsistencia temporal dos projetos

In [ ]:
df_projetos.head()

,projeto_id,cliente_id,nome_projeto,tipo_servico,squad,status,data_inicio,data_fim_prevista,data_fim_real,horas_vendidas,valor_contrato
0,PRJ0001,CLI013,Dashboard Bi - Windsor,Dashboard B.I.,Delta,Concluído,2024-03-19,2024-05-06,2024-05-08,288,58987.08
1,PRJ0002,CLI022,Diagnóstico De Dados - Umbu,Diagnóstico De Dados,Alpha,Cancelado,2024-10-01,2024-10-28,NaT,89,24044.51
2,PRJ0003,CLI051,Pipeline De Dados - Cristal,Pipeline De Dados,Charlie,Concluído,2025-06-16,2025-10-20,2025-10-15,423,91525.58
3,PRJ0004,CLI060,Dashboard Bi - Kairós,Dashboard Bi,Echo,Concluído,2024-03-19,2024-04-25,2024-04-26,220,47409.01
4,PRJ0005,CLI020,Modelo Preditivo - Duna,Modelo Preditivo,Echo,Concluído,2024-12-09,2025-02-27,2025-03-12,338,61970.49


In [ ]:
# @title ## 5.4 Dataset - df_apontamentos

In [ ]:
# @title ## Informações Iniciais do Dataset - df_apontamentos

display(Markdown('### **Primeiras Linhas do Dataset**'))
display(df_apontamentos.head())

display(Markdown('### **Informações do Dataset**'))
display(df_apontamentos.info())

display(Markdown('### **Resumo Estatístico do Dataset**'))
display(df_apontamentos.describe().T)

display(Markdown('### **Quantidade de Valores Ausentes**'))
display(df_apontamentos.isnull().sum())

display(Markdown('### **Quantidade de Valores Duplicados**'))
display(df_apontamentos.duplicated().sum())

display(Markdown('### **Quantidade de Valores Únicos**'))
display(df_apontamentos.nunique())

display(Markdown('### **Os tipos de dados**'))
display(df_apontamentos.dtypes)

### **Primeiras Linhas do Dataset**

,apontamento_id,projeto_id,analista_id,data,horas,atividade
0,APT002473,PRJ0032,ANL006,2026-03-23,6,Reunião com cliente
1,APT004798,PRJ0064,ANL008,2026-05-08,6.5,Apresentação
2,APT005566,PRJ0079,ANL012,2025-06-02,6,Coleta
3,APT000099,PRJ0003,ANL009,2025-09-15,4.5,Documentação
4,APT009367,PRJ0134,ANL009,2026-01-22,4,Modelagem


### **Informações do Dataset**

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9879 entries, 0 to 9878
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   apontamento_id  9879 non-null   object
 1   projeto_id      9879 non-null   object
 2   analista_id     9879 non-null   object
 3   data            9879 non-null   object
 4   horas           9879 non-null   object
 5   atividade       9879 non-null   object
dtypes: object(6)
memory usage: 463.2+ KB


None

...

np.int64(80)

### **Quantidade de Valores Únicos**

,0
apontamento_id,9799
projeto_id,140
analista_id,24
data,2037
horas,23
atividade,15


### **Os tipos de dados**

,0
apontamento_id,object
projeto_id,object
analista_id,object
data,object
horas,object
atividade,object


In [ ]:
# @title Padronização das colunas
df_apontamentos = padronizar_nomes_colunas(df_apontamentos)
df_apontamentos = remover_espacos(df_apontamentos)
df_apontamentos = padronizar_strings(df_apontamentos)

In [ ]:
# converter a coluna
df_apontamentos['apontamento_id'] = df_apontamentos['apontamento_id'].str.upper()
df_apontamentos['projeto_id'] = df_apontamentos['projeto_id'].str.upper()
df_apontamentos['analista_id'] = df_apontamentos['analista_id'].str.upper()
df_apontamentos.head()

,apontamento_id,projeto_id,analista_id,data,horas,atividade
0,APT002473,PRJ0032,ANL006,2026-03-23,6,Reunião Com Cliente
1,APT004798,PRJ0064,ANL008,2026-05-08,6.5,Apresentação
2,APT005566,PRJ0079,ANL012,2025-06-02,6,Coleta
3,APT000099,PRJ0003,ANL009,2025-09-15,4.5,Documentação
4,APT009367,PRJ0134,ANL009,2026-01-22,4,Modelagem


In [ ]:
# @title ## Tratar os dados duplicados
# verificar os dados duplicados
df_apontamentos.duplicated().sum()

np.int64(80)

In [ ]:
duplicated_rows = df_apontamentos[df_apontamentos.duplicated()]
display(duplicated_rows)

,apontamento_id,projeto_id,analista_id,data,horas,atividade
900,APT001262,PRJ0014,ANL008,2024-08-15,5,Modelagem
1123,APT004172,PRJ0055,ANL014,10/07/2024,4,Dashboard
2399,APT007470,PRJ0106,ANL003,2026-06-25,6.5,Testes
2590,APT004560,PRJ0062,ANL011,2024-12-12,3,Apresentação
2863,APT006888,PRJ0098,ANL012,09/04/2025,7,Modelagem
...,...,...,...,...,...,...
9701,APT002426,PRJ0031,ANL003,2024-10-23,4,Apresentação
9750,APT003978,PRJ0051,ANL005,2024-11-18,4,Modelagem
9859,APT007276,PRJ0104,ANL012,2024-05-21,4,Modelagem
9867,APT000486,PRJ0007,ANL010,2024-12-19,2,Modelagem


In [ ]:
# remover os dados duplicados
df_apontamentos = df_apontamentos.drop_duplicates()

In [ ]:
# converter string horas e numeros

df_apontamentos['horas'] = converter_numeros(df_apontamentos['horas'])

In [ ]:
df_apontamentos.dtypes

,0
apontamento_id,string[python]
projeto_id,string[python]
analista_id,string[python]
data,string[python]
horas,Float64
atividade,string[python]


In [ ]:
# @title ## Converter string em datetime
# converter as data
df_apontamentos['data'] = converter_datas_mistas(
    df_apontamentos['data']
)

In [ ]:
df_apontamentos['atividade'].unique()

<StringArray>
['Reunião Com Cliente',        'Apresentação',              'Coleta',
        'Documentação',           'Modelagem',              'Testes',
                 'Etl',           'Dashboard']
Length: 8, dtype: string

In [ ]:
df_apontamentos.head()

,apontamento_id,projeto_id,analista_id,data,horas,atividade
0,APT002473,PRJ0032,ANL006,2026-03-23,6.0,Reunião Com Cliente
1,APT004798,PRJ0064,ANL008,2026-05-08,65.0,Apresentação
2,APT005566,PRJ0079,ANL012,2025-06-02,6.0,Coleta
3,APT000099,PRJ0003,ANL009,2025-09-15,45.0,Documentação
4,APT009367,PRJ0134,ANL009,2026-01-22,4.0,Modelagem


In [ ]:
# @title ## 5.5 Dataset df_satisfacao


In [ ]:
# @title ## Informações Iniciais do Dataset - df_satisfacao
display(Markdown('### **Primeiras Linhas do Dataset**'))
display(df_satisfacao.head())

display(Markdown('### **Informações do Dataset**'))
display(df_satisfacao.info())

display(Markdown('### **Resumo Estatístico do Dataset**'))
display(df_satisfacao.describe().T)

display(Markdown('### **Quantidade de Valores Ausentes**'))
display(df_satisfacao.isnull().sum())

display(Markdown('### **Quantidade de Valores Duplicados**'))
display(df_satisfacao.duplicated().sum())

display(Markdown('### **Quantidade de Valores Únicos**'))
display(df_satisfacao.nunique())

display(Markdown('### **Os tipos de dados**'))
display(df_satisfacao.dtypes)

### **Primeiras Linhas do Dataset**

,pesquisa_id,projeto_id,data_pesquisa,nota_nps,comentario
0,PSQ0001,PRJ0001,2024-05-27,NaN,"Resultado ok, prazo apertado."
1,PSQ0002,PRJ0003,2025-10-20,10.0,Documentação impecável.
2,PSQ0003,PRJ0004,2024-05-08,9.0,Documentação impecável.
3,PSQ0004,PRJ0005,2025-03-27,7.0,"Bom projeto, comunicação pode melhorar."
4,PSQ0005,PRJ0006,17/07/2025,6.0,Suporte demorou a responder.


### **Informações do Dataset**

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   pesquisa_id    103 non-null    object 
 1   projeto_id     103 non-null    object 
 2   data_pesquisa  103 non-null    object 
 3   nota_nps       95 non-null     float64
 4   comentario     103 non-null    object 
dtypes: float64(1), object(4)
memory usage: 4.2+ KB


None

...

np.int64(0)

### **Quantidade de Valores Únicos**

,0
pesquisa_id,103
projeto_id,103
data_pesquisa,100
nota_nps,8
comentario,10


### **Os tipos de dados**

,0
pesquisa_id,object
projeto_id,object
data_pesquisa,object
nota_nps,float64
comentario,object


In [ ]:
# @title ## Padronização das colunas
df_satisfacao = padronizar_nomes_colunas(df_satisfacao)
df_satisfacao = remover_espacos(df_satisfacao)
df_satisfacao = padronizar_strings(df_satisfacao)
#

In [ ]:
# converter a coluna
df_satisfacao['pesquisa_id'] = df_satisfacao['pesquisa_id'].str.upper()
df_satisfacao['projeto_id'] = df_satisfacao['projeto_id'].str.upper()
df_satisfacao.head()

,pesquisa_id,projeto_id,data_pesquisa,nota_nps,comentario
0,PSQ0001,PRJ0001,2024-05-27,NaN,"Resultado Ok, Prazo Apertado."
1,PSQ0002,PRJ0003,2025-10-20,10.0,Documentação Impecável.
2,PSQ0003,PRJ0004,2024-05-08,9.0,Documentação Impecável.
3,PSQ0004,PRJ0005,2025-03-27,7.0,"Bom Projeto, Comunicação Pode Melhorar."
4,PSQ0005,PRJ0006,17/07/2025,6.0,Suporte Demorou A Responder.


In [ ]:
# @title ## Tratamentos dos dados nulos

nulls_nota_nps = df_satisfacao[df_satisfacao['nota_nps'].isnull()]
display(nulls_nota_nps)

,pesquisa_id,projeto_id,data_pesquisa,nota_nps,comentario
0,PSQ0001,PRJ0001,2024-05-27,NaN,"Resultado Ok, Prazo Apertado."
14,PSQ0015,PRJ0020,2024-04-11,NaN,O Painel Virou Rotina Da Diretoria.
28,PSQ0029,PRJ0037,2025-06-27,NaN,O Painel Virou Rotina Da Diretoria.
42,PSQ0043,PRJ0057,2026-05-11,NaN,Entrega No Prazo E Time Muito Atencioso.
56,PSQ0057,PRJ0074,2025-11-17,NaN,Entrega No Prazo E Time Muito Atencioso.
70,PSQ0071,PRJ0092,2024-09-05,NaN,O Painel Virou Rotina Da Diretoria.
84,PSQ0085,PRJ0110,2025-03-03,NaN,Documentação Impecável.
98,PSQ0099,PRJ0134,2026-02-26,NaN,"Resultado Ok, Prazo Apertado."


> Os registros com nota_nps ausente possuem comentários preenchidos. Portanto, a ausência da nota não representa necessariamente ausência de participação na pesquisa. Como não é possível inferir a nota quantitativa a partir do texto do comentário sem introduzir uma regra subjetiva, os valores ausentes foram preservados.

In [ ]:
# quantificar o problema
df_satisfacao['nota_nps'].isna().sum()

np.int64(8)

In [ ]:
percentual_nulos = (
    df_satisfacao["nota_nps"].isna().mean() * 100
)

percentual_nulos

np.float64(7.766990291262135)

> 7,77% das pesquisas não possuem nota NPS registrada.

In [ ]:
df_satisfacao.groupby(
    df_satisfacao["nota_nps"].isna()
).size()

,0
nota_nps,
False,95
True,8


In [ ]:
df_satisfacao.merge(
    df_projetos[
        ["projeto_id", "status", "tipo_servico", "squad"]
    ],
    on="projeto_id",
    how="left"
)

,pesquisa_id,projeto_id,data_pesquisa,nota_nps,comentario,status,tipo_servico,squad
0,PSQ0001,PRJ0001,2024-05-27,NaN,"Resultado Ok, Prazo Apertado.",Concluído,Dashboard B.I.,Delta
1,PSQ0002,PRJ0003,2025-10-20,10.0,Documentação Impecável.,Concluído,Pipeline De Dados,Charlie
2,PSQ0003,PRJ0004,2024-05-08,9.0,Documentação Impecável.,Concluído,Dashboard Bi,Echo
3,PSQ0004,PRJ0005,2025-03-27,7.0,"Bom Projeto, Comunicação Pode Melhorar.",Concluído,Modelo Preditivo,Echo
4,PSQ0005,PRJ0006,17/07/2025,6.0,Suporte Demorou A Responder.,Concluído,Pipeline De Dados,Bravo
...,...,...,...,...,...,...,...,...
98,PSQ0099,PRJ0134,2026-02-26,NaN,"Resultado Ok, Prazo Apertado.",Concluído,Dashboard Bi,Charlie
99,PSQ0100,PRJ0135,2024-04-19,10.0,O Painel Virou Rotina Da Diretoria.,Concluído,Dashboard Bi,Bravo
100,PSQ0101,PRJ0137,2025-07-22,9.0,O Painel Virou Rotina Da Diretoria.,Concluído,Treinamento,Alpha
101,PSQ0102,PRJ0139,2024-12-23,7.0,"Bom Projeto, Comunicação Pode Melhorar.",Concluído,Modelo Preditivo,Echo


In [ ]:
# @title ## Salvar os datasets processados
df_clientes.to_csv('/content/drive/MyDrive/Colab Notebooks/mini_projeto/data/processed/df_clientes_processed.csv', index=False)
df_analistas.to_csv('/content/drive/MyDrive/Colab Notebooks/mini_projeto/data/processed/df_analistas_processed.csv', index=False)
df_projetos.to_csv('/content/drive/MyDrive/Colab Notebooks/mini_projeto/data/processed/df_projetos_processed.csv', index=False)
df_apontamentos.to_csv('/content/drive/MyDrive/Colab Notebooks/mini_projeto/data/processed/df_apontamentos_processed.csv', index=False)
df_satisfacao.to_csv('/content/drive/MyDrive/Colab Notebooks/mini_projeto/data/processed/df_satisfacao_processed.csv', index=False)
#